# Certified workflows

Each workflow is self-contained; run its sections in order. See the [tutorial index](../tutorials/index.md) and [result contracts](../reference/result-contracts.md).


## Finite metric and variation proofs

<!-- workflow-id: finite-metric-proofs -->

Can I certify an optimal finite bound and a total variation?

Compute exact finite Lipschitz constants and jump variation, then replay their proofs.


### Certify a finite Lipschitz constant

A finite domain needs all pairwise slopes. The constant below is 2 on {-1,0,2}; it is not the interval derivative bound on [-1,2].


In [ ]:
from dataclasses import replace

import sympy as sp

import funcprops as fp

x = sp.Symbol("x", real=True)
metric = fp.lipschitz(x * x, x, domain=sp.FiniteSet(-1, 0, 2), return_result=True)
assert metric.constant == 2 and metric.optimal is fp.TruthValue.TRUE
assert metric.verify() and fp.verify_certificate(metric.proof)


### Replay total variation

The quadratic contributes motion 2, and the value 10 at zero contributes two excursions of size 10. Total variation is 22.


In [ ]:
spike = sp.Piecewise((10, sp.Eq(x, 0)), (x * x, True))
variation = fp.variation(spike, x, domain=sp.Interval(-1, 1), return_result=True)
assert variation.total_variation == 22 and variation.verify()
assert not fp.verify_certificate(
    replace(variation.proof, total_variation=sp.Integer(2))
)


### Separate a proved value from replay support

An affine interval query has optimal constant 3. This route carries no standalone finite-domain replay proof; a useful mathematical answer and supported replay are separate questions.


In [ ]:
affine = fp.lipschitz(3 * x + 2, x, return_result=True)
assert affine.constant == 3 and affine.optimal is fp.TruthValue.TRUE
assert affine.proof is None and affine.verify() is False


Expected: The finite constant is 2, the spike variation is 22, and both supplied proofs replay.

All finite pairs and all owned excursions contribute. A changed total is rejected.

Scope: Interval derivative bounds can be useful without the finite replay contract.


## Piecewise partitions

<!-- workflow-id: piecewise-partitions -->

How can I reuse branch results without losing jumps?

Reuse cells, exact images and owned seams; coverage and global order have different premises.


### Partition a function with a jump

A jump can preserve strict order. A reusable partition records the input ownership of each interval and all seam values.


In [ ]:
import sympy as sp

import funcprops as fp

x = sp.Symbol("x", real=True)
f = sp.Piecewise((x, x < 0), (x + 2, True))
p = fp.monotonic_partition(f, x, domain=sp.Interval(-1, 1), return_result=True)
assert p.complete and not p.critical_points_complete
assert p.same_strict_direction() is fp.Monotonicity.STRICTLY_INCREASING
image = fp.range_from_monotonic_partition(f, x, p).image
assert image == sp.Union(sp.Interval.Ropen(-1, 0), sp.Interval(2, 3))


### Inspect an owned point

The image is [-1,0) union [2,3]. Replacing the value at zero with 100 adds an isolated image point and breaks global increasing order.


In [ ]:
spike = sp.Piecewise((100, sp.Eq(x, 0)), (x, True))
s = fp.monotonic_partition(spike, x, domain=sp.Interval(-1, 1), return_result=True)
assert s.complete and s.same_strict_direction() is None
assert fp.range_from_monotonic_partition(spike, x, s).image == sp.Union(
    sp.Interval(-1, 1) - sp.FiniteSet(0), sp.FiniteSet(100)
)


### Keep undefined points separate

A requested interval containing an original source hole is not certified as a full partition by this route. Partition completeness also does not assert completeness of the derivative zero set.


In [ ]:
held = sp.Mul(x, sp.Pow(x, -1, evaluate=False), evaluate=False)
holey = sp.Piecewise((held, x < 1), (x, True), evaluate=False)
assert not fp.monotonic_partition(
    holey, x, domain=sp.Interval(-1, 2), return_result=True
).complete


Expected: The image is [-1,0) union [2,3]; the owned spike breaks strict order.

A complete cell cover is separate from global order and derivative-root completeness.

Scope: Source holes and unresolved seams can prevent partition completion.


## Restricted domains and held expressions

<!-- workflow-id: restricted-domains -->

How do input restrictions and held operations affect an answer?

Real intervals and finite domains; held syntax preserves holes.


In [ ]:
import sympy as sp

import funcprops as fp

x, y = sp.symbols("x y", real=True)
image = fp.range(sp.exp(x), x, y, constraints=(x > 0) & (x < 1))
assert sp.simplify(image ^ ((y > 1) & (y < sp.E))) is sp.false
held = sp.Mul(x, sp.Pow(x, -1, evaluate=False), evaluate=False)
record = fp.track_domain(held, x, sp.Interval(-1, 1))
assert record.verify() and record.complete
assert record.effective == sp.Union(sp.Interval.Ropen(-1, 0), sp.Interval.Lopen(0, 1))
assert sp.simplify(held) == 1
assert record.effective.contains(0) is sp.false
assert fp.sign(x - 1, x, domain=sp.FiniteSet(0, 2)) is fp.Sign.SIGN_CHANGING

# Holding a logarithm preserves its base as well as its source restrictions.
decreasing_log = sp.log(x, sp.Rational(1, 2), evaluate=False)
assert fp.domain(decreasing_log, x) == (x > 0)
assert (
    fp.monotonicity(decreasing_log, x, domain=sp.Interval(2, 4))
    is fp.Monotonicity.STRICTLY_DECREASING
)
enclosure = fp.certified_real_ball(decreasing_log, x, sp.Interval(2, 4))
assert enclosure.verify()
assert enclosure.lower <= -2 < -1 <= enclosure.upper


power_domain = fp.track_domain(x ** sp.sqrt(2), x, sp.Interval(-1, 2))
assert power_domain.verify() and power_domain.natural_complete
assert power_domain.natural == sp.Interval(0, sp.oo)
assert power_domain.effective == sp.Interval(0, 2)
guarded = sp.Piecewise((1, sp.log(x) > 0), (0, True))
guard_domain = fp.track_domain(guarded, x, sp.Interval(-1, 2))
assert guard_domain.verify() and guard_domain.effective == sp.Interval.Lopen(0, 2)
assert (
    fp.continuous(decreasing_log, x, domain=sp.Interval(2, 4))
    is fp.Continuity.CONTINUOUS
)
assert fp.lipschitz(decreasing_log, x, domain=sp.Interval(2, 4)) == 1 / (2 * sp.log(2))


Expected: The exponential image is (1,E); held x/x excludes zero; positive irrational powers retain zero.

Source definition is tracked before cancellation and before branch rules.

Scope: Holding cannot recover operands already simplified by SymPy before the query.


## Algebraic backend roots

<!-- workflow-id: algebraic-roots -->

How does the algebraic backend preserve principal branches and holes?

Supported algebraic enumeration with principal-branch and raw-source filtering.


In [ ]:
import sympy as sp

import funcprops as fp

z = sp.Symbol("z")
rectangle = (-2 - 2 * sp.I, 2 + 2 * sp.I)
r = fp.complex_zeros((z - 1) ** 3, z, rectangle=rectangle, return_result=True)
assert r.verify() and r.complete and r.algebraic_certificate is not None
assert r.root_count == 1 and r.total_count == 3
held = sp.Mul(z**2, sp.Pow(z, -1, evaluate=False), evaluate=False)
hole = fp.isolate_complex_roots(held, z, rectangle)
assert hole.verify() and hole.complete and hole.roots == () and hole.total_count == 0
radical = fp.certify_algebraic_complex_roots(sp.sqrt(z), z, rectangle)
assert radical.verify() and radical.complete and radical.roots == (0,)
assert radical.root_count == 1 and radical.multiplicity_count is None


Expected: A cubic zero has multiplicity three; a held hole removes a candidate; sqrt(z) has unresolved source multiplicity.

Candidate coverage and source multiplicity use separate contracts.

Scope: Exact root locations do not automatically establish source multiplicity for radicals.


## Budgets and resumption

<!-- workflow-id: budgets-and-resume -->

What can I retain when a bounded search stops?

Checked states and same-environment trusted checkpoint transport; in-memory checkpoint contract.


In [ ]:
import pickle
from dataclasses import replace

import sympy as sp

import funcprops as fp

x = sp.Symbol("x", real=True)
policy = fp.EnclosurePolicy(max_cells=400, max_depth=9)
seed = fp.adaptive_root_coverage(
    sp.exp(x) - 2, x, sp.Interval(0, 1), policy=replace(policy, max_cells=0)
)
assert seed.verify() and not seed.complete and seed.root_count is None
assert seed.state.exhaustion == "cell-budget"
state = pickle.loads(pickle.dumps(seed.state))  # trusted, same-environment transport
resumed = fp.resume_root_coverage(state, policy=policy)
assert resumed.verify() and resumed.complete and resumed.root_count == 1
z = sp.Symbol("z")
run = fp.adaptive_quotient_isolation(
    1 / (sp.exp(z) - 2), z, -1 - sp.I, 1 + sp.I, stages=0
)
assert run.verify() and not run.complete and run.state.unresolved
assert run.termination_contract.verify() and run.termination_contract.complete is False


unfinished = fp.domain(x ** sp.sqrt(2), x, work_budget=1e-6)
assert unfinished.complete is False and unfinished.certified is False
assert unfinished.operation == "domain" and unfinished.source == "work-budget-exhausted"
known = fp.track_domain(x ** sp.sqrt(2), x, sp.Interval(0, 1), work_budget=10)
assert known.verify() and known.effective == sp.Interval(0, 1)


Expected: An unresolved real seed resumes to one root; the cofinal quotient seed retains its unresolved region.

A resource limit stops a search without proving exclusion.

Scope: Checkpoints are trusted same-environment records, not a durable untrusted-file format.


## Certified extrema bounds

<!-- workflow-id: certified-extrema-bounds -->

Can I bound extrema without solving the stationary equation exactly?

Finite real components, implicit critical points, and separate coverage/value accuracy.


In [ ]:
import sympy as sp

import funcprops as fp

x = sp.Symbol("x", real=True)
policy = fp.EnclosurePolicy(
    precision=20, max_precision=40, max_refinements=1, max_cells=160, max_depth=7
)
proof = fp.certify_extrema_bounds(
    sp.exp(x) + x * x,
    x,
    sp.Interval(-1, 1),
    tolerance=sp.Rational(1, 10000),
    policy=policy,
)
assert proof.verify() and proof.complete
assert proof.coverage_complete and proof.accuracy_complete
assert proof.minimum_bounds[1] - proof.minimum_bounds[0] <= proof.tolerance
assert proof.maximum_bounds[1] - proof.maximum_bounds[0] <= proof.tolerance
assert not proof.exact_complete
assert fp.verify_certificate(proof)


Expected: Both value brackets have width at most 1/10000, while exact extrema remain unresolved.

Value accuracy can be certified without a closed-form stationary coordinate.

Scope: The supported finite smooth branches and interval enclosures do not cover arbitrary nonsmooth inputs.


## Exact algebraic coefficients

<!-- workflow-id: algebraic-coefficients -->

How do I keep an algebraic coefficient on its intended real conjugate?

Use a uniquely isolated generator in a bounded real number field and retain source holes.


### Represent an exact coefficient field

An isolating interval selects sqrt(2) from the two roots of a rational polynomial. The generator guard prevents its negative conjugate from entering the query.


In [ ]:
import sympy as sp

import funcprops as fp

x, y = sp.symbols("x y", real=True)
field = fp.encode_algebraic_coefficients((sp.sqrt(2) * x, x < sp.sqrt(2)), (x,))
assert field.verify() and sp.degree(field.polynomial, field.generator) == 2
image = fp.range(sp.sqrt(2) * x, x, y, constraints=(x > 0) & (x < 1))
assert sp.simplify(image ^ ((y > 0) & (y < sp.sqrt(2)))) is sp.S.false


### Preserve an original hole

The held quotient agrees with 1 wherever it is defined. Its range at the excluded point sqrt(2) is empty.


In [ ]:
base = x - sp.sqrt(2)
raw = sp.Mul(base, sp.Pow(base, -1, evaluate=False), evaluate=False)
assert fp.range(raw, x, y, constraints=sp.Eq(x, sp.sqrt(2))) is sp.S.false


### Recognize the field boundary

The supported compositum has degree at most four and at most four nonrational constants. Larger fields decline this encoding; other property routes may still apply.


In [ ]:
assert (
    fp.encode_algebraic_coefficients((sp.sqrt(2) + sp.sqrt(3) + sp.sqrt(5),), (x,))
    is None
)


Expected: The image is (0,sqrt(2)); a canceled source hole has empty image at its excluded point.

The rational generator guard selects a unique real embedding.

Scope: The compositum degree is at most four, with at most four nonrational constants and no extra coefficient parameters in the lift.


## Guarded and Piecewise results

<!-- workflow-id: guarded-piecewise -->

How do parameter guards and singleton branches reach downstream results?

Finite real affine coefficients, supported branch coverage and singleton seams.


In [ ]:
import sympy as sp

import funcprops as fp

x, a, b = sp.symbols("x a b", real=True)
cases = fp.solve(a * x + b, x, domain=sp.S.Reals, guarded=True, return_certificate=True)
assert cases.verify() and cases.complete and len(cases.cases) == 3
assert cases.specialize(sp.Ne(a, 0)).root_count == 1
assert (
    fp.monotonicity(a * x, x, assumptions=a > 0) is fp.Monotonicity.STRICTLY_INCREASING
)
f = sp.Piecewise((x, x < 0), (100, sp.Eq(x, 0)), (x + 10, True))
proof = fp.certify_property_coverage(f, x, sp.Interval(-1, 1))
assert proof.verify() and proof.complete
assert proof.conclusion("maximum").value == (100, sp.FiniteSet(0))
image = proof.conclusion("range").value.doit()
assert image.contains(100) is sp.true and image.contains(0) is sp.false


Expected: The affine equation has three guarded cases, and the Piecewise maximum is 100 at zero.

First-match ownership and parameter hypotheses are mathematical premises.

Scope: Unresolved guards or missing seams cannot supply unconditional conclusions.


## Independent native replay

<!-- workflow-id: replay -->

Can I check a native proof with its proposer disabled?

Native witness checking without search; factory-based and algebraic backend trust differs.


In [ ]:
from dataclasses import replace
from unittest.mock import patch

import sympy as sp

import funcprops as fp

x = sp.Symbol("x", real=True)
proof = fp.prove_root_coverage(sp.exp(x) - 2, x, sp.Interval(0, 1))
assert proof.verify() and proof.complete


def forbidden(*args, **kwargs):
    raise AssertionError("replay invoked proposing search")


with patch("funcprops.coverage_search.propose_leaf", forbidden):
    assert fp.verify_certificate(proof)
assert not fp.verify_certificate(replace(proof, root_count=77))
assert not fp.verify_certificate(proof, expr=sp.exp(x) - 3, var=x)
assert fp.replay_status(proof).outcome == "complete-coverage"
assert fp.replay_status(object()).certified is None


Expected: The original proof replays; the altered total and foreign source fail.

Registered concrete proof types bind the theorem to its original context.

Scope: Factory-based and algebraic routes have their own declared replay dependencies.


## Polynomial parameter geometry

<!-- workflow-id: parameter-geometry -->

How many distinct real roots does a polynomial family have?

Partition a bounded polynomial family by distinct real-root counts and retain unresolved regions.


### Separate root-count strata

For x**2+a*x+b, the discriminant separates zero, one and two distinct real roots. The count at a repeated root is one, not its multiplicity two.


In [ ]:
import sympy as sp

import funcprops as fp

x, a, b = sp.symbols("x a b", real=True)
proof = fp.certify_parameter_geometry(x * x + a * x + b, x, (a, b), max_cells=100)
assert proof.verify() and proof.complete
assert proof.specialize({a: 0, b: -1}) == 2
assert proof.specialize({a: 2, b: 1}) == 1
assert proof.specialize({a: 0, b: 1}) == 0


### Resume an unfinished partition

With no cell work, the exact unresolved frontier covers the whole query. It is valid partial evidence. Resumption discharges that frontier without changing the source or parameter domain.


In [ ]:
partial = fp.certify_parameter_geometry(x * x + a * x + b, x, (a, b), max_cells=0)
assert partial.verify() and not partial.complete
assert partial.specialize({a: 0, b: -1}) is None
assert fp.resume_parameter_geometry(partial, max_cells=100).complete


### Respect parameter assumptions and scope

A positive parameter cannot be specialized to zero or a negative value. This interface supports rational polynomial families of degree at most four; it does not decompose arbitrary transcendental families.


In [ ]:
p = sp.Symbol("p", positive=True)
guarded = fp.certify_parameter_geometry(x * x - p, x, (p,), max_cells=100)
assert guarded.specialize({p: 1}) == 2 and guarded.specialize({p: -1}) is None
try:
    fp.certify_parameter_geometry(sp.exp(x) - a, x, (a,))
except ValueError:
    pass
else:
    raise AssertionError("transcendental family admitted to polynomial geometry")


Expected: The demonstrated quadratic strata have counts 2, 1 and 0; the partial frontier resumes to completion.

Counts refer to distinct roots and exact guarded regions, including degree loss.

Scope: Degree is at most four in the root coordinate, with up to four real parameters and rational interval endpoints. Transcendental families are outside this interface.


## Real root coverage

<!-- workflow-id: real-root-coverage -->

When is a real-root count exhaustive?

Compact witnesses, rational multiplicities, recognized periodic families and tails.


In [ ]:
import sympy as sp

import funcprops as fp

x = sp.Symbol("x", real=True)
compact = fp.adaptive_root_coverage(sp.exp(x) - 2, x, sp.Interval(0, 1))
assert compact.verify() and compact.complete and compact.root_count == 1
multiple = fp.prove_root_coverage((x * x - 2) ** 3, x, sp.Interval(-2, 2))
assert (
    multiple.verify() and multiple.root_count == 2 and multiple.multiplicity_count == 6
)
open_domain = fp.adaptive_root_coverage(sp.exp(x) - 1, x, sp.Interval.open(0, 1))
assert open_domain.complete and open_domain.root_count == 0
periodic = fp.infinite_root_coverage(sp.sin(x), x)
assert periodic.verify() and periodic.complete and periodic.root_count is sp.oo
tails = fp.infinite_root_coverage(sp.exp(x) - x * x, x)
assert tails.verify() and tails.complete and tails.root_count == 1


Expected: The compact exponential has one root; the repeated polynomial has two distinct roots of total multiplicity six.

Open endpoints, periodic indexed families and tail proofs complete different parts of the domain.

Scope: A compact root count cannot cover an infinite domain without exhaustive families or tails.


## Unsupported and partial results

<!-- workflow-id: unsupported-and-partial -->

How should I interpret unsupported or unfinished isolation?

Resource exhaustion and boundary obstruction do not prove exclusion.


In [ ]:
import sympy as sp

import funcprops as fp

z = sp.Symbol("z")
seed = fp.certify_complex_isolation(
    sp.sin(z), z, -4 - sp.I, 4 + sp.I, policy=fp.EnclosurePolicy(max_cells=0)
)
assert seed.verify() and seed.unresolved and not seed.coverage_complete
assert seed.total_count is None and not seed.complete
boundary = fp.certify_complex_isolation(
    sp.exp(z) - 1,
    z,
    -sp.I,
    1 + sp.I,
    policy=fp.EnclosurePolicy(max_cells=32, max_depth=1),
    max_boundary_depth=1,
)
assert boundary.verify() and not boundary.complete and boundary.total_count is None
unsupported = fp.certify_complex_isolation(sp.log(z), z, -1 - sp.I, 1 + sp.I)
assert not unsupported.complete


Expected: The unresolved seed and boundary obstruction retain unknown totals and incomplete status.

Successful local replay does not assert exhaustive coverage.

Scope: Unsupported grammar and unclear boundaries are neither root-exclusion proofs nor mathematical counterexamples.


## Complex isolation and completion

<!-- workflow-id: complex-isolation -->

Does my complex result isolate roots individually at the requested accuracy?

Parameter-free entire grammar on rational rectangles with zero-free outer boundaries.


In [ ]:
import sympy as sp

import funcprops as fp

z = sp.Symbol("z")
sine = fp.certify_complex_isolation(sp.sin(z), z, -4 - sp.I, 4 + sp.I)
assert sine.verify() and sine.complete and sine.root_count == 3
nonreal = fp.complex_zeros(
    sp.exp(z) + 1, z, rectangle=(-1 - 4 * sp.I, 1 + 4 * sp.I), return_result=True
)
assert nonreal.verify() and nonreal.root_count == nonreal.multiplicity_count == 2
cluster = fp.certify_complex_isolation(
    sp.sin(z) ** 2,
    z,
    -1 - sp.I,
    1 + sp.I,
    policy=fp.EnclosurePolicy(max_depth=0),
    use_multiple=False,
)
assert cluster.verify() and cluster.coverage_complete and not cluster.complete
assert cluster.multiplicity_count == 2 and cluster.root_count is None
assert not cluster.isolation_complete and not cluster.accuracy_complete
accurate = fp.certify_complex_isolation(
    sp.exp(z) - 1,
    z,
    -sp.Rational(1, 10) - sp.I / 10,
    sp.Rational(1, 10) + sp.I / 10,
    max_box_diameter=sp.Rational(1, 1000),
    policy=fp.EnclosurePolicy(max_depth=32, max_cells=1024),
)
assert accurate.verify() and accurate.complete and accurate.accuracy_complete


Expected: Three sine zeros and two non-real exponential zeros are counted; a retained cluster is valid partial evidence.

Coverage, individual isolation and requested box accuracy are separate claims.

Scope: The native route requires its declared parameter-free grammar, rational rectangles and clear outer boundary.


## Count-preserving Newton refinement

<!-- workflow-id: newton-refinement -->

Can I shrink a root box without discarding its complement?

A checked simple base, certified contraction and all complement strips.


In [ ]:
import sympy as sp

import funcprops as fp

z = sp.Symbol("z")
radius = sp.Rational(1, 10)
parent = fp.certify_analytic_contour(
    sp.exp(z) - 1, z, -radius - sp.I * radius, radius + sp.I * radius
)
assert parent.verify() and parent.zero_count == 1
refined = fp.certify_complex_refinement(parent)
assert refined is not None and refined.verify() and fp.verify_certificate(refined)
assert len(refined.regions) == 5
assert sum(c.zero_count for c in refined.regions) == 1


Expected: Five child/complement regions preserve total zero count one.

Certified contraction accelerates an already checked local count.

Scope: A small candidate box alone does not prove isolation or discard the surrounding region.


## Finite branch-chart gluing

<!-- workflow-id: branch-chart-gluing -->

Can local logarithms cover a loop without a global logarithm?

Checked logarithm transitions and selected power branches; nontrivial cycles remain explicit.


In [ ]:
import sympy as sp

import funcprops as fp

z = sp.Symbol("z")
regions = (
    fp.CoverageRectangle(-2 - sp.I / 4, 2 - sp.I / 8),
    fp.CoverageRectangle(-2 + sp.I / 8, 2 + sp.I / 4),
    fp.CoverageRectangle(-2 - sp.I / 4, -sp.Rational(7, 4) + sp.I / 4),
    fp.CoverageRectangle(sp.Rational(7, 4) - sp.I / 4, 2 + sp.I / 4),
)
policy = fp.EnclosurePolicy(
    precision=20, max_precision=40, max_refinements=1, max_cells=160, max_depth=8
)
logarithm = fp.certify_log_atlas(z, z, regions, policy=policy)
assert logarithm.verify() and logarithm.coverage_complete
assert logarithm.obstructions and not logarithm.complete
square_log = fp.certify_log_atlas(z * z, z, regions, policy=policy)
power = fp.GluedPowerProof(square_log, sp.S.Half)
assert power.verify() and power.complete
assert fp.verify_certificate(power)
# This selects a glued square root; it does not redefine principal sqrt(z**2).


Expected: The logarithm has full chart coverage and an obstruction; the selected square root of z**2 glues.

Logarithm sheet transitions and power multipliers have different completion conditions.

Scope: Finite log/power atlases do not supply a general chart-aware contour grammar.


## Gluing local branch recipes

<!-- workflow-id: branch-recipes -->

Do my selected local solutions agree on every overlap?

Assemble supplied holomorphic recipes, check every overlap and distinguish disagreement from a global obstruction.


### Select local solutions

Give formulas for (w-z)**2=z on a rectangle away from zero. Logarithm sheets 0 and 2 produce the same selected square root.


In [ ]:
from dataclasses import replace

import sympy as sp

import funcprops as fp

z, w, u = sp.symbols("z w u")
rectangle = fp.CoverageRectangle(1 - sp.I / 4, 2 + sp.I / 4)
logarithm = fp.certify_log_atlas(z, z, (rectangle,)).charts[0]
relation = (w - z) ** 2 - z
left = fp.certify_branch_chart(
    relation, z, w, rectangle, z + sp.exp(u / 2), logarithms=((u, logarithm),)
)
right = fp.certify_branch_chart(
    relation,
    z,
    w,
    rectangle,
    z + sp.exp(u / 2),
    logarithms=((u, replace(logarithm, sheet=2)),),
)
glued = fp.glue_branch_charts((left, right))
assert glued.verify() and glued.coverage_complete and glued.gluing_complete


### Interpret disagreement correctly

Sheet 1 selects the opposite square root. The supplied charts disagree. This does not prove that every possible choice of local branches fails to glue on this rectangle.


In [ ]:
opposite = replace(right, logarithms=((u, replace(logarithm, sheet=1)),))
disagreement = fp.glue_branch_charts((left, opposite))
assert disagreement.verify() and not disagreement.complete and disagreement.obstructions


### An anchor alone is insufficient

Two solutions of an implicit equation can meet at an isolated point. The formulas z and -z both satisfy w**2=z**2 and agree at zero, yet their derivatives differ, so the gluing checker keeps this overlap unresolved.


In [ ]:
box = fp.CoverageRectangle(-1 - sp.I, 1 + sp.I)
a = fp.certify_branch_chart(w * w - z * z, z, w, box, z)
b = fp.certify_branch_chart(w * w - z * z, z, w, box, -z)
crossing = fp.glue_branch_charts((a, b))
assert crossing.verify() and not crossing.complete and crossing.unresolved_overlaps


Expected: The compatible square-root recipes glue; opposite selected recipes disagree; crossing branches stay unresolved.

Agreement at one anchor does not prove equality of local formulas.

Scope: This assembles supplied recipes for entire implicit relations; it does not discover arbitrary implicit branches.


## Implicit common zeros and orders

<!-- workflow-id: implicit-common-orders -->

Can I prove common zeros and their orders at an implicitly located root?

Congruence/radical identities and local orders, separately from exhaustive root coverage.


In [ ]:
import sympy as sp

import funcprops as fp

z = sp.Symbol("z")
g = sp.exp(z) - 2
rectangle = (sp.Rational(69, 100) - sp.I / 100, sp.Rational(7, 10) + sp.I / 100)
policy = fp.EnclosurePolicy(
    precision=20, max_precision=40, max_refinements=1, max_cells=192
)
base = fp.certify_analytic_contour(g, z, *rectangle, policy=policy)
f = sp.sin(g) + z * (sp.exp(g) - 1)
h = sp.sin(g) ** 2 + z * (sp.exp(g) - 1) ** 2
common = fp.certify_common_zero((f, h), base, max_order=3, policy=policy)
assert common.verify() and common.common_zero_status is fp.TruthValue.TRUE
assert common.orders_complete and common.multiplicities == (1, 2)
assert not common.coverage_complete and not common.complete
assert fp.verify_certificate(common)
# Exact local order does not exclude other zeros elsewhere in the rectangle.


Expected: The local multiplicities are (1,2); this local theorem is not exhaustive root coverage.

Exact congruences and nonzero jets establish local orders.

Scope: Unproved identities and possible additional zeros keep the corresponding completion fields false.


## Quotients, holes and poles

<!-- workflow-id: quotients-and-holes -->

Which counted points are source zeros, excluded extension zeros or poles?

Certified entire quotients; shared orders require exact identities and exhaustive counts.


In [ ]:
import sympy as sp

import funcprops as fp

z = sp.Symbol("z")
g = sp.exp(z) - 2
rectangle = (-1 - sp.I, 1 + sp.I)
policy = fp.EnclosurePolicy(max_cells=8192, max_depth=24, max_taylor_order=5)
held = sp.Mul(g**2, g, sp.Pow(g, -1, evaluate=False), evaluate=False)
hole = fp.certify_quotient_isolation(held, z, *rectangle, policy=policy)
assert hole.verify() and hole.complete and hole.total_count == 0
assert hole.proof.extension_zero_count == hole.proof.excluded_zero_multiplicity == 2
assert hole.proof.source.guards == (g,)
for denominator, poles in ((sp.sin(g), 0), (sp.sin(g) ** 2, 1)):
    q = fp.certify_quotient_isolation(g / denominator, z, *rectangle, policy=policy)
    assert q.verify() and q.complete and q.total_count == 0
    assert q.pole_multiplicity_count == poles
    assert q.proof.winding_count == -poles


Expected: The held cancellation has zero source roots and extension multiplicity two; the examples have zero and one pole.

Original denominator restrictions survive meromorphic extension.

Scope: Unknown shared orders or unclear boundaries prevent the corresponding complete ledger.


## Selected local branches

<!-- workflow-id: local-branches -->

How does a selected logarithmic germ affect a local residue?

Explicit local logarithm values; no global chart-gluing claim.


In [ ]:
from dataclasses import replace

import sympy as sp

import funcprops as fp

z = sp.Symbol("z")
chart = fp.LocalBranchContext(-1, ((z, sp.I * sp.pi),))
residue = fp.residue(
    sp.log(z) / (z + 1), z, -1, branch_context=chart, return_certificate=True
)
assert residue is not None and residue.verify() and residue.residue == sp.I * sp.pi
assert not replace(residue, branch_context=None).verify()
assert fp.residue(sp.log(z) / (z + 1), z, -1, return_certificate=True) is None


Expected: The selected residue is I*pi; removing the branch context fails replay.

A local germ supplies source-bound branch values at a particular point.

Scope: A local branch calculation is not a global principal-branch identity.
